#Set up and Load the Data

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os
print(os.listdir('/content/drive/MyDrive/Capstone Project'))

['WDICSV.csv', 'WDICountry.csv']


In [14]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Capstone Project/WDICSV.csv')
country_meta = pd.read_csv('/content/drive/MyDrive/Capstone Project/WDICountry.csv')

print(df.shape)
print(country_meta.shape)
print(df.head())

(396970, 70)
(264, 31)
                  Country Name Country Code  \
0  Africa Eastern and Southern          AFE   
1  Africa Eastern and Southern          AFE   
2  Africa Eastern and Southern          AFE   
3  Africa Eastern and Southern          AFE   
4  Africa Eastern and Southern          AFE   

                                      Indicator Name     Indicator Code  1960  \
0  Access to clean fuels and technologies for coo...     EG.CFT.ACCS.ZS   NaN   
1  Access to clean fuels and technologies for coo...  EG.CFT.ACCS.RU.ZS   NaN   
2  Access to clean fuels and technologies for coo...  EG.CFT.ACCS.UR.ZS   NaN   
3            Access to electricity (% of population)     EG.ELC.ACCS.ZS   NaN   
4  Access to electricity, rural (% of rural popul...  EG.ELC.ACCS.RU.ZS   NaN   

   1961  1962  1963  1964  1965  ...       2016       2017       2018  \
0   NaN   NaN   NaN   NaN   NaN  ...  18.685118  19.205632  19.742772   
1   NaN   NaN   NaN   NaN   NaN  ...   7.606712   7.926604   

#Filter the Indicator

In [15]:
indicator_codes = [
    'SE.ADT.LITR.ZS',    # Literacy rate
    'SE.SEC.ENRR',       # Secondary enrollment
    'SE.TER.ENRR',       # Tertiary enrollment
    'IT.NET.USER.ZS',    # Internet users %
    'IT.CEL.SETS.P2',    # Mobile subscriptions
    'IT.NET.BBND.P2',    # Broadband subscriptions
    'NY.GDP.PCAP.CD',    # GDP per capita
    'NY.GDP.MKTP.KD.ZG', # GDP growth %
    'SL.UEM.TOTL.ZS'     # Unemployment %
]

df_filtered = df[df['Indicator Code'].isin(indicator_codes)].copy()
print(df_filtered.shape)
print(df_filtered['Indicator Code'].unique())

(2385, 70)
['IT.NET.BBND.P2' 'NY.GDP.MKTP.KD.ZG' 'NY.GDP.PCAP.CD' 'IT.NET.USER.ZS'
 'SE.ADT.LITR.ZS' 'IT.CEL.SETS.P2' 'SE.SEC.ENRR' 'SE.TER.ENRR'
 'SL.UEM.TOTL.ZS']


#Unpivot the columns

In [16]:
id_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
year_cols = [col for col in df_filtered.columns if col.isdigit()]

df_long = df_filtered.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name='Year',
    value_name='Value'
)

df_long['Year'] = df_long['Year'].astype(int)

print(df_long.shape)
print(df_long.head())
print(df_long['Year'].min(), "-", df_long['Year'].max())

(157410, 6)
                  Country Name Country Code  \
0  Africa Eastern and Southern          AFE   
1  Africa Eastern and Southern          AFE   
2  Africa Eastern and Southern          AFE   
3  Africa Eastern and Southern          AFE   
4  Africa Eastern and Southern          AFE   

                                      Indicator Name     Indicator Code  Year  \
0     Fixed broadband subscriptions (per 100 people)     IT.NET.BBND.P2  1960   
1                              GDP growth (annual %)  NY.GDP.MKTP.KD.ZG  1960   
2                       GDP per capita (current US$)     NY.GDP.PCAP.CD  1960   
3   Individuals using the Internet (% of population)     IT.NET.USER.ZS  1960   
4  Literacy rate, adult total (% of people ages 1...     SE.ADT.LITR.ZS  1960   

        Value  
0         NaN  
1         NaN  
2  186.089515  
3         NaN  
4         NaN  
1960 - 2025


In [17]:
#Restrict year range
df_long = df_long[(df_long['Year'] >= 2000) & (df_long['Year'] <= 2023)]
df_long = df_long.dropna(subset=['Value'])

print(df_long.shape)
print(df_long['Year'].min(), "-", df_long['Year'].max())

(44683, 6)
2000 - 2023


In [18]:
#Remove aggregate countries
real_country_codes = country_meta['Country Code'].unique()
df_long = df_long[df_long['Country Code'].isin(real_country_codes)]

print(df_long.shape)
print("Number of unique countries remaining:", df_long['Country Code'].nunique())

(44683, 6)
Number of unique countries remaining: 264


In [19]:
print(country_meta[['Country Code', 'Short Name', 'Region']].head(15))
print("\nRows with blank Region (likely aggregates):", country_meta['Region'].isna().sum())

   Country Code                   Short Name                      Region
0           ABW                        Aruba   Latin America & Caribbean
1           AFE  Africa Eastern and Southern                         NaN
2           AFG                  Afghanistan  Middle East & North Africa
3           AFW   Africa Western and Central                         NaN
4           AGO                       Angola          Sub-Saharan Africa
5           ALB                      Albania       Europe & Central Asia
6           AND                      Andorra       Europe & Central Asia
7           ARB                   Arab World                         NaN
8           ARE         United Arab Emirates  Middle East & North Africa
9           ARG                    Argentina   Latin America & Caribbean
10          ARM                      Armenia       Europe & Central Asia
11          ASM               American Samoa         East Asia & Pacific
12          ATG          Antigua and Barbuda   Lati

In [20]:
# Get only real countries (those with a non-null Region)
real_countries = country_meta[country_meta['Region'].notna()]
real_country_codes = real_countries['Country Code'].unique()

df_long = df_long[df_long['Country Code'].isin(real_country_codes)]

print(df_long.shape)
print("Number of unique countries remaining:", df_long['Country Code'].nunique())

(35690, 6)
Number of unique countries remaining: 217


In [21]:
#Remove duplicates
before = df_long.shape[0]
df_long = df_long.drop_duplicates(subset=['Country Code', 'Indicator Code', 'Year'])
after = df_long.shape[0]

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 35690
Rows after: 35690
Duplicates removed: 0


In [22]:
#Pivot indicators into columns
df_indicators = df_long.pivot_table(
    index=['Country Code', 'Year'],
    columns='Indicator Name',
    values='Value'
).reset_index()

df_indicators.columns.name = None

print(df_indicators.shape)
print(df_indicators.head())

(5176, 11)
  Country Code  Year  Fixed broadband subscriptions (per 100 people)  \
0          ABW  2000                                             NaN   
1          ABW  2001                                             NaN   
2          ABW  2002                                             NaN   
3          ABW  2003                                        1.515545   
4          ABW  2004                                        7.469988   

   GDP growth (annual %)  GDP per capita (current US$)  \
0               7.622921                  20681.023027   
1               4.182002                  20740.132583   
2              -0.944953                  21307.248251   
3               1.110505                  21949.485996   
4               7.293728                  23700.631990   

   Individuals using the Internet (% of population)  \
0                                         15.442823   
1                                         17.100000   
2                                         

In [25]:
country_lookup = country_meta[['Country Code', 'Short Name', 'Region', 'Income Group']].copy()

# Keep only countries that actually appear in our indicators table
country_lookup = country_lookup[country_lookup['Country Code'].isin(df_indicators['Country Code'])]

# Drop rows with no Region (shouldn't be any left, but just in case)
country_lookup = country_lookup.dropna(subset=['Region'])

# Create the Africa vs Rest of World flag
country_lookup['Region_Group'] = country_lookup['Region'].apply(
    lambda x: 'Africa' if x == 'Sub-Saharan Africa' else 'Rest of World'
)

country_lookup = country_lookup.rename(columns={'Short Name': 'Country Name'})

print(country_lookup.shape)
print(country_lookup.head())
print("\nRegion Group counts:")
print(country_lookup['Region_Group'].value_counts())

(217, 5)
  Country Code Country Name                      Region         Income Group  \
0          ABW        Aruba   Latin America & Caribbean          High income   
2          AFG  Afghanistan  Middle East & North Africa           Low income   
4          AGO       Angola          Sub-Saharan Africa  Lower middle income   
5          ALB      Albania       Europe & Central Asia  Upper middle income   
6          AND      Andorra       Europe & Central Asia          High income   

    Region_Group  
0  Rest of World  
2  Rest of World  
4         Africa  
5  Rest of World  
6  Rest of World  

Region Group counts:
Region_Group
Rest of World    169
Africa            48
Name: count, dtype: int64


In [24]:
print(country_meta.columns.tolist())

['Country Code', 'Short Name', 'Table Name', 'Long Name', '2-alpha code', 'Currency Unit', 'Special Notes', 'Region', 'Income Group', 'WB-2 code', 'National accounts base year', 'National accounts reference year', 'SNA price valuation', 'Lending category', 'Other groups', 'System of National Accounts', 'Alternative conversion factor', 'PPP survey year', 'Balance of Payments Manual in use', 'External debt Reporting status', 'System of trade', 'Government Accounting concept', 'IMF data dissemination standard', 'Latest population census', 'Latest household survey', 'Source of most recent Income and expenditure data', 'Vital registration complete', 'Latest agricultural census', 'Latest industrial data', 'Latest trade data', 'Latest water withdrawal data']


In [26]:
# Keep only countries that exist in BOTH tables
valid_codes = country_lookup['Country Code'].unique()
df_indicators = df_indicators[df_indicators['Country Code'].isin(valid_codes)]

print("Final indicators shape:", df_indicators.shape)
print("Final country count:", df_indicators['Country Code'].nunique())

# Double check nothing is missing
missing = set(df_indicators['Country Code']) - set(country_lookup['Country Code'])
print("Any Country Codes in indicators but missing from lookup:", missing)

Final indicators shape: (5176, 11)
Final country count: 217
Any Country Codes in indicators but missing from lookup: set()


In [27]:
df_indicators.to_csv('/content/drive/MyDrive/Capstone Project/wdi_indicators_clean.csv', index=False)
country_lookup.to_csv('/content/drive/MyDrive/Capstone Project/country_lookup.csv', index=False)

print("Done. Two files saved to your Capstone Project folder:")
print(" - wdi_indicators_clean.csv (fact table)")
print(" - country_lookup.csv (dimension/lookup table)")

Done. Two files saved to your Capstone Project folder:
 - wdi_indicators_clean.csv (fact table)
 - country_lookup.csv (dimension/lookup table)
